# CMMD final-test evaluation

In [1]:
# @@ENV@@ — initialize TensorFlow and import the final-test evaluator
import os
import sys
from pathlib import Path

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
BASE_PATH = Path(os.environ.get("DUM_BASE", os.environ.get("ISIC24_BASE", "/mnt/disk1/home/cgiovannoni/mod-attendance")))
sys.path.insert(0, str(BASE_PATH / "src"))

import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

import pandas as pd
from IPython.display import display
from isic24 import paths as pth
from isic24 import statistical_evaluation as se
from isic24.xai_subset import CENTRAL_SETTINGS

pth.DATASET, pth.VARIANT = "cmmd", "processed"
ART_PATH = pth.art_path()
OUTPUT_DIR = ART_PATH / "statest"


2026-11-12 11:04:44.440655: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794477884.453813 2806899 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794477884.457764 2806899 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
I0000 00:00:1794477887.062064 2806899 gpu_device.cc:2413] Ignoring visible gpu device (device: 1, name: NVIDIA GeForce GT 1030, pci bus id: 0000:65:00.0, compute capability: 6.1) with core count: 3. The minimum required count is 8. You can adjust this requirement with the env var TF_MIN_GPU_MULTIPROCESSOR_COUNT.


In [ ]:
# @@PROTOCOL@@ — freeze reference and validation-selected configurations before test inference
STUDIES = {
    "vanilla": "20261015-114125",
    "fixed_scaling": "20261016-234343",
    "learnable_scaling": "20261020-224124",
    "loss_regularization": "20261019-233600",
    "controlled_fusion": "20261020-063421",
    "adaptive_gating": "20261020-020650",
}
SEEDS = [8, 47, 53, 71, 90, 101]
EXPECTED_SETTINGS = {
    "vanilla": 1, "fixed_scaling": 9, "learnable_scaling": 5,
    "loss_regularization": 5, "controlled_fusion": 1, "adaptive_gating": 5,
}
protocol = se.freeze_protocol(
    {name: ART_PATH / "runs" / name / study for name, study in STUDIES.items()},
    ART_PATH, OUTPUT_DIR, SEEDS, EXPECTED_SETTINGS, CENTRAL_SETTINGS,
    n_boot=2000, n_permutations=5000, random_seed=2026,
)
selection = pd.DataFrame(protocol["evaluations"])
display(selection.groupby(["panel", "strategy", "setting_tag", "delta"])
        .agg(n_seeds=("seed", "size"), validation_auc=("selection_val_auc", "mean")))


KeyboardInterrupt: 

: 

In [ ]:
# @@TEST_PREDICTIONS@@ — validate patient separation and predict the untouched test cohort
test_frame, predictions = se.predict_test(protocol, OUTPUT_DIR)
print(f"[TEST] {len(test_frame)} images, {test_frame.patient_id.nunique()} patients")


I0000 00:00:1794392526.191420 2230317 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22800 MB memory:  -> device: 0, name: Quadro RTX 6000, pci bus id: 0000:17:00.0, compute capability: 7.5
I0000 00:00:1794392542.277283 2230317 cuda_dnn.cc:529] Loaded cuDNN version 90300


[TEST] 1/78 adaptive_gating:base:d+0.000:s8
[TEST] 2/78 adaptive_gating:base:d+0.000:s47
[TEST] 3/78 adaptive_gating:base:d+0.000:s53
[TEST] 4/78 adaptive_gating:base:d+0.000:s71
[TEST] 5/78 adaptive_gating:base:d+0.000:s90
[TEST] 6/78 adaptive_gating:base:d+0.000:s101
[TEST] 7/78 controlled_fusion:base:d+0.000:s8
[TEST] 8/78 controlled_fusion:base:d+0.250:s8
[TEST] 9/78 controlled_fusion:base:d+0.000:s47
[TEST] 10/78 controlled_fusion:base:d+0.250:s47
[TEST] 11/78 controlled_fusion:base:d+0.000:s53
[TEST] 12/78 controlled_fusion:base:d+0.250:s53
[TEST] 13/78 controlled_fusion:base:d+0.000:s71
[TEST] 14/78 controlled_fusion:base:d+0.250:s71
[TEST] 15/78 controlled_fusion:base:d+0.000:s90
[TEST] 16/78 controlled_fusion:base:d+0.250:s90
[TEST] 17/78 controlled_fusion:base:d+0.000:s101
[TEST] 18/78 controlled_fusion:base:d+0.250:s101
[TEST] 19/78 fixed_scaling:a0p500:d+0.000:s8
[TEST] 20/78 fixed_scaling:a0p500:d+0.000:s47
[TEST] 21/78 fixed_scaling:a0p500:d+0.000:s53
[TEST] 22/78 fixed_s

In [ ]:
# @@STATISTICS@@ — patient-paired confidence intervals, block-swap tests and seed sign tests
results = se.analyze_test(protocol, test_frame, predictions)


[STATISTICS] Patient bootstrap complete; computing paired contrasts
[STATISTICS] reference: 17 paired comparisons complete
[STATISTICS] selected: 17 paired comparisons complete


In [ ]:
# @@RESULTS@@ — inspect predictive estimates and prespecified paired contrasts
display(results["summary"].round(5))
display(results["contrasts"].round(5))
print("Differences = strategy minus baseline: higher AUC and lower Brier/ECE are favorable.")
print("95% intervals cover patient sampling conditional on fitted seeds; seed SD is separate.")
print("Intervals are pointwise; p-values use Holm correction within each panel and test unit.")


,panel,strategy,metric,mean,seed_sd,seed_variance,ci_lo,ci_hi,n_seeds,n_images,n_patients
0,reference,adaptive_gating,auc,0.69722,0.00864,0.00007,0.65028,0.73765,6,748,355
1,reference,adaptive_gating,brier,0.23544,0.02032,0.00041,0.21066,0.26024,6,748,355
2,reference,adaptive_gating,ece,0.18260,0.02826,0.00080,0.16203,0.21475,6,748,355
3,reference,controlled_fusion,auc,0.69677,0.00991,0.00010,0.65117,0.73721,6,748,355
4,reference,controlled_fusion,brier,0.26283,0.04444,0.00197,0.23996,0.28709,6,748,355
5,reference,controlled_fusion,ece,0.21710,0.06248,0.00390,0.19745,0.25042,6,748,355
6,reference,fixed_scaling,auc,0.72343,0.00903,0.00008,0.67703,0.76385,6,748,355
7,reference,fixed_scaling,brier,0.23271,0.02260,0.00051,0.21081,0.25704,6,748,355
8,reference,fixed_scaling,ece,0.17733,0.04339,0.00188,0.15575,0.21404,6,748,355
9,reference,image_only_control,auc,0.69899,0.02175,0.00047,0.65421,0.73953,6,748,355


,panel,baseline,strategy,metric,difference,ci_lo,ci_hi,favorable_seeds,n_informative_seeds,patient_swap_p,seed_sign_p,patient_swap_p_holm,seed_sign_p_holm
0,reference,vanilla,adaptive_gating,auc,-0.00854,-0.02309,0.00640,2,6,0.37852,0.68750,1.00000,1.0
1,reference,vanilla,adaptive_gating,brier,-0.00832,-0.01941,0.00215,3,6,0.12677,1.00000,1.00000,1.0
2,reference,vanilla,adaptive_gating,ece,-0.00679,-0.02586,0.00882,3,6,0.40492,1.00000,1.00000,1.0
3,reference,vanilla,controlled_fusion,auc,-0.00898,-0.02237,0.00426,1,6,0.18396,0.21875,1.00000,1.0
4,reference,vanilla,controlled_fusion,brier,0.01908,0.01072,0.02689,2,6,0.00020,0.68750,0.01020,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,selected,table_only_control,loss_regularization,brier,0.05492,0.03238,0.07927,0,6,0.00020,0.03125,0.01020,1.0
98,selected,table_only_control,loss_regularization,ece,0.13559,0.07964,0.16255,0,6,0.00020,0.03125,0.01020,1.0
99,selected,table_only_control,vanilla,auc,0.04928,-0.00019,0.09644,6,6,0.04259,0.03125,0.77584,1.0
100,selected,table_only_control,vanilla,brier,0.04677,0.02489,0.07062,0,6,0.00060,0.03125,0.02280,1.0


Differences = strategy minus baseline: higher AUC and lower Brier/ECE are favorable.
95% intervals cover patient sampling conditional on fitted seeds; seed SD is separate.
Intervals are pointwise; p-values use Holm correction within each panel and test unit.


In [ ]:
# @@EXPORT@@ — save final-test tables, provenance and the selected-panel confidence-interval plot
se.save_results(results, protocol, OUTPUT_DIR)
print(f"[SAVED] Final-test results: {OUTPUT_DIR}")


[SAVED] Final-test results: /mnt/disk1/home/cgiovannoni/mod-attendance/artifacts/cmmd/processed/statest


: 